# Reddit Suspicious User ID Stripping 

This notebook will strip the IDs of all suspicious users in posts that do not mention bots. These IDs need to be collected so they can be connected to other posts and comments they have been involved in.

In [ ]:
# ARCHIVED: copy of suspicious_id_collection_v1.ipynb from an older 'cleaning' branch. Same steps, but it saves
# the account lookups as three pickles (active, suspended, deleted). Replaced by collect_accused_usernames_and_first_mention.ipynb.
# Libraries.
#import libraries
import pandas as pd
import numpy as np
import os 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
from string import punctuation
import functools
import operator
from dotenv import load_dotenv
from pmaw import PushshiftAPI
import praw
import prawcore
from prawcore.exceptions import NotFound
import pprint
import ast
import pprint
import os 
import itertools
import pickle

In [ ]:
# Every r/TheseFuckingAccounts submission, 2014-2022.
#import post ('submissions') data
csv='/data/Emily_data/TheseFuckingAccounts_submissions.csv'


In [ ]:
#create df
subs=pd.read_csv(csv)

Dropping NA values from selftext which is the post content. We checked for duplicates of posts in previous notebook and know there aren't any so we don't need to do that here. 

In [ ]:
# Keep posts with body text (link posts have none).
posts=subs.dropna(subset=['selftext'])

Remove bots

In [ ]:
# Drop posts that mention 'bot' anywhere in title or text (substring match, so 'robot' etc. go too).
posts=posts[~posts.selftext.str.contains("(?i)bot")]
posts=posts[~posts.title.str.contains("(?i)bot")]

In [ ]:
# Drop removed/deleted bodies.
remove = ['[removed]','[deleted]']

posts=posts.loc[posts.selftext.isin(remove) == False]

In [ ]:
# Extract u/name and user/name mentions (lower-cased) from text and title; no accused/victim distinction yet.
#only looking for user names this time around 
user_pat = "(?:u/|user/)([A-Za-z0-9_-]+)"
def find_text(df, col_name, pat, new_col):
    df[new_col] = df[col_name].apply(str.lower).str.findall(pat)
    
    return df
    
#need to take names from title and post text  

posts=find_text(posts, 'selftext', user_pat, 'users1')
posts=find_text(posts, 'title', user_pat, 'users2')
posts['users']=posts['users1']+posts['users2']

Make a list of individual users so we can pull dates and number of times they appear at a later date

In [ ]:
# Distinct names per post, then one flat list over all posts (a name repeats once per post that names it).
unique_users=[list(set(users)) for users in posts.users]

In [ ]:
unique_user_list=functools.reduce(operator.iconcat, unique_users, [])

## API call setup 

In [ ]:
# Load Reddit API credentials from a .env file outside the repo.
%load_ext dotenv

In [ ]:
%dotenv /home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env

In [ ]:
# Reddit API client. The .env file names the login key login_name, so username here resolves to None.
reddit_api = praw.Reddit(
client_id=os.getenv("praw_script"),
client_secret=os.getenv("praw_key"),
username=os.getenv("user_name"),
password=os.getenv("password"),
user_agent= os.getenv("user_agent"),ratelimit_seconds=360)

## Collect User ID and Status 

In [ ]:
# Look up each name on Reddit: Active, Suspended or Deleted, with the raw profile for active/suspended accounts.
# Up to 10 tries on rate limits; any other error stops the run.
def account_data(list_of_unique_usernames):
    user_data={}
    for x in list_of_unique_usernames:
        #init of dict with x as key[0]
        user_data[x]={}
        #retries for failure
        retry_count = 0
        while retry_count < 10:
            try:
                #get info on account 
                redditor_info = reddit_api.redditor(str(x))
                #print(f"Fetching data for: {redditor_info.name}")
                
                try:
                    #check attributes profiles have diff meta data depending on status
                    if hasattr(redditor_info, 'is_suspended') == True:
                        y='Suspended'
                    elif hasattr(redditor_info, 'id') == True:
                        y='Active'
                    else:
                        y='Deleted'
                except prawcore.exceptions.NotFound:
                    y='Deleted'
                #if user is still active grab and nest data 
                if y == 'Active' or y == 'Suspended':
                    #print('Yes')
                    user_data[x][y]=redditor_info.__dict__
                #if not active just record it
                else:
                    user_data[x][y]='No Data'
                    #print('None')
            #handle errors
            except prawcore.exceptions.TooManyRequests as e:
                print(f"Querying user {y} failed with {e} retry count currently {retry_count}")
                retry_count += 1
                if retry_count == 10:
                    raise e
            except Exception as e:
                print (f"Hit unexpected error with class {e.__class__}:{e}")
                raise e
            else:
                # Cheeky way to bypass doing this work again
                retry_count = 10
        
    return user_data
            
            
                
               

In [ ]:
# Slow: one API call per name.
user_dict=account_data(unique_user_list)


In [ ]:
# Split the lookups into three tables by outcome, with profile fields as columns.
def dict_to_dfs(nested_dict):
    '''transform nested dict into separate dfs based
    on outcome since available data is different for 
    each subgroup.'''
    #level one keys are user names so those become the first col
    df=pd.DataFrame(nested_dict.keys(),columns=['User_Names'])
    #make some empty lists to add as cols later
    outcome_list=[]
    user_data_dicts=[]
    #dig into the nested dict and append values to empty list
    for key1,value1 in nested_dict.items():
        for key2,value2 in value1.items():
            outcome_list.append(key2)
            user_data_dicts.append(value2)
    #make new cols in df
    df['Account_Outcome']=outcome_list
    df['User_Data_Dict']=user_data_dicts
    #split the dfs because they have different data attached
    #def df_split_tidy()
    drop_cols=['User_Data_Dict','_reddit','subreddit']
    active_df=df[df['Account_Outcome']=='Active']
    active_df=active_df.join(pd.json_normalize(active_df['User_Data_Dict'])).drop(columns=drop_cols,errors='ignore')
    sus_df=df[df['Account_Outcome']=='Suspended']
    sus_df=sus_df.join(pd.json_normalize(sus_df['User_Data_Dict'])).drop(columns=drop_cols,errors='ignore')
    deleted_df=df[df['Account_Outcome']=='Deleted']
    

    return active_df,sus_df,deleted_df
        
        
        

### Change column drop and save to functional format! 

In [ ]:
active_df,sus_df,deleted_df=dict_to_dfs(user_dict)

In [ ]:
# Save each table as a pickle in the working folder.
file_path='sus_user_data.pickle'
with open(file_path, 'wb') as file:
    pickle.dump(sus_df, file)
    

In [ ]:
file_path='active_user_data.pickle'
with open(file_path, 'wb') as file:
    pickle.dump(active_df, file)

In [ ]:
file_path='deleted_user_data.pickle'
with open(file_path, 'wb') as file:
    pickle.dump(deleted_df, file)